<a href="https://colab.research.google.com/github/Nayab189/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check


## 1. Build the feature vector

**Feature Engineering Strategy & Data Transformation**
---
We extract historical daily data from March 2026 (1 March–31 March 2026) using DuckDB. To ensure **reproducibility**, we apply deterministic ordering (`ORDER BY content_id LIMIT 100000`). We integrate traditional GSC search metrics alongside **all 5 multi-channel AI referral streams** (ChatGPT, Claude, Gemini, Perplexity, Copilot) with strict zero-imputation via `COALESCE`. Survivorship bias is resolved by tracking vanished pages as extreme decline labels (`is_declining = 1`).

In [ ]:
import duckdb
import pandas as pd
from google.colab import userdata

# Safely fetch HF_TOKEN from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Query with deterministic ordering and full multi-channel AI feature extraction
df = con.sql(f"""
    WITH feature_window AS (
        SELECT
            content_hash_id AS content_id,
            BOOL_OR(gsc_data_available) AS is_available,
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            CASE
                WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions))
                ELSE 0.0
            END AS ctr_30d,
            AVG(gsc_sum_position) AS avg_position,
            COALESCE(SUM(sessions_ai), 0) AS ai_sessions_30d,
            COALESCE(SUM(ai_chatgpt), 0) AS ai_chatgpt_30d,
            COALESCE(SUM(ai_perplexity), 0) AS ai_perplexity_30d,
            COALESCE(SUM(ai_gemini), 0) AS ai_gemini_30d,
            COALESCE(SUM(ai_copilot), 0) AS ai_copilot_30d,
            COALESCE(SUM(ai_claude), 0) AS ai_claude_30d
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    label_window AS (
        SELECT
            content_hash_id AS content_id,
            SUM(gsc_impressions) AS future_impressions_30d
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY content_hash_id
    )
    SELECT
        f.content_id,
        '2026-03-31' AS snapshot_date,
        f.is_available,
        f.impressions_30d,
        f.clicks_30d,
        f.ctr_30d,
        f.avg_position,
        f.ai_sessions_30d,
        f.ai_chatgpt_30d,
        f.ai_perplexity_30d,
        f.ai_gemini_30d,
        f.ai_copilot_30d,
        f.ai_claude_30d,
        COALESCE(c.word_count, 800) AS word_count,
        COALESCE(DATE_DIFF('day', CAST(c.content_created_date AS DATE), DATE '2026-03-31'), 90) AS content_age_days,

        -- Robust Decline Label with Survivorship Bias Mitigation
        CASE
            WHEN l.content_id IS NULL THEN 1  -- Completely vanished in next window = Extreme Decline
            WHEN l.future_impressions_30d < (f.impressions_30d * 0.8) THEN 1
            ELSE 0
        END AS is_declining

    FROM feature_window f
    LEFT JOIN label_window l ON f.content_id = l.content_id
    LEFT JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_id = c.content_hash_id
    ORDER BY f.content_id  -- Enforces deterministic reproducibility across runs
    LIMIT 100000
""").df()

print("Feature Vector successfully constructed with deterministic ordering!")
print(f"Shape of Feature Matrix: {df.shape}")
print(df.head(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature Vector successfully constructed with deterministic ordering!
Shape of Feature Matrix: (100000, 16)
                 content_id snapshot_date  is_available  impressions_30d  \
0  content_000005d4ced12088    2026-03-31          True             86.0   
1  content_00007bd2985b77c3    2026-03-31          True             47.0   
2  content_0000cd28fbda69f3    2026-03-31          True             29.0   

   clicks_30d  ctr_30d  avg_position  ai_sessions_30d  ai_chatgpt_30d  \
0         0.0      0.0    258.291667              0.0             0.0   
1         0.0      0.0     10.826087              0.0             0.0   
2         0.0      0.0      8.538462              0.0             0.0   

   ai_perplexity_30d  ai_gemini_30d  ai_copilot_30d  ai_claude_30d  \
0                0.0            0.0             0.0            0.0   
1                0.0            0.0             0.0            0.0   
2                0.0            0.0             0.0            0.0   

   word_count 

## 2. Feature notes (meaning, missing, categorical, available-when?)

* **impressions_30d / clicks_30d**: Aggregated search visibility metrics. Zero-imputed. Available before decision moment.
* **ctr_30d**: Click-through rate. Available before decision moment.
* **avg_position**: Unweighted average of daily rank sums (Note: serves as a directional baseline indicator due to daily aggregate summation scaling quirks). Available before decision moment.
* **AI Channels (ChatGPT, Claude, Gemini, Perplexity, Copilot, Sessions)**: Multi-channel referral volumes. Modeled as sparse indicator features. Available before decision moment.
* **word_count & content_age_days**: Dimensional content features. Available at snapshot date.

In [ ]:
# Updated feature columns including ALL 5 AI channels and aggregate sessions
feature_cols = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position",
    "ai_sessions_30d",
    "ai_chatgpt_30d",
    "ai_claude_30d",
    "ai_gemini_30d",
    "ai_copilot_30d",
    "ai_perplexity_30d",
    "word_count",
    "content_age_days"
]
label_col = "is_declining"
context_cols = ["content_id", "snapshot_date"]

# 1. Null Check Verification
null_summary = df[context_cols + feature_cols + [label_col]].isnull().sum()
print("Null Value Verification across Feature Vector:")
print(null_summary)

# 2. Class Balance Check for Target Label (Addressing Claude's recommendation)
class_balance = df[label_col].value_counts(normalize=True) * 100
print("\n--- Target Label Class Balance (is_declining %) ---")
print(class_balance)

print("\nSummary Statistics of Feature Matrix:")
print(df[feature_cols].describe())

Null Value Verification across Feature Vector:
content_id           0
snapshot_date        0
impressions_30d      0
clicks_30d           0
ctr_30d              0
avg_position         0
ai_sessions_30d      0
ai_chatgpt_30d       0
ai_claude_30d        0
ai_gemini_30d        0
ai_copilot_30d       0
ai_perplexity_30d    0
word_count           0
content_age_days     0
is_declining         0
dtype: int64

--- Target Label Class Balance (is_declining %) ---
is_declining
1    53.308
0    46.692
Name: proportion, dtype: float64

Summary Statistics of Feature Matrix:
       impressions_30d     clicks_30d        ctr_30d   avg_position  \
count    100000.000000  100000.000000  100000.000000  100000.000000   
mean       1583.272690       4.642090       0.004513     626.700493   
std        5202.312697      23.233015       0.036818    2772.248235   
min           1.000000       0.000000       0.000000       0.000000   
25%          19.000000       0.000000       0.000000      20.818182   
50%    

## 3. The leakage hunt

We evaluate clean feature correlations against our target label. **Finding on Correlation:** Individual features show very weak linear correlations with the target. This is expected since traffic decay is typically driven by non-linear, multi-feature interactions rather than isolated linear associations—a nuance that tree-based models will capture during the modeling phase.

Additionally, we test and verify our leakage detector by injecting an intentional post-snapshot feature (`TRAP_future_post_snapshot_clicks`), observing an artificial score spike (-1.0/near-perfect correlation), and immediately purging it.

In [ ]:
# 1. Baseline honest feature correlations with Target
honest_correlations = df[feature_cols].apply(lambda col: col.corr(df[label_col]))
print("--- HONEST FEATURE CORRELATIONS ---")
print(honest_correlations)

# 2. INJECTING THE LEAKAGE TRAP (Future window feature simulation)
df['TRAP_future_post_snapshot_clicks'] = df[label_col].apply(lambda target: 0 if target == 1 else 150)
trap_correlation = df['TRAP_future_post_snapshot_clicks'].corr(df[label_col])

print("\n--- LEAKAGE TRAP TEST ---")
print(f"Artificial Trap Feature Correlation with Target: {trap_correlation:.4f}")
print("Notice: Artificial post-snapshot leakage causes near-perfect predictability, proving our leakage audit works!")

# 3. CLEANUP: Dropping the contaminated feature
df.drop(columns=['TRAP_future_post_snapshot_clicks'], inplace=True)
print("\nContaminated leakage column successfully removed! Feature matrix restored to honest state.")

--- HONEST FEATURE CORRELATIONS ---
impressions_30d     -0.039463
clicks_30d          -0.074092
ctr_30d              0.011197
avg_position        -0.017397
ai_sessions_30d      0.000981
ai_chatgpt_30d       0.000593
ai_claude_30d        0.001825
ai_gemini_30d        0.004521
ai_copilot_30d       0.003834
ai_perplexity_30d   -0.007592
word_count          -0.099131
content_age_days     0.082307
dtype: float64

--- LEAKAGE TRAP TEST ---
Artificial Trap Feature Correlation with Target: -1.0000
Notice: Artificial post-snapshot leakage causes near-perfect predictability, proving our leakage audit works!

Contaminated leakage column successfully removed! Feature matrix restored to honest state.


## 4. What I excluded and why

* **post_snapshot_impressions_90d / future_ga4_sessions**: Excluded completely to prevent future-window temporal data leakage.
* **client_tenant_id / raw_target_url**: Excluded strictly to ensure workspace-level data privacy compliance.

In [ ]:
excluded_fields_check = [
    "post_snapshot_impressions_90d",
    "client_tenant_id",
    "raw_target_url",
    "future_ga4_sessions",
]

print("Privacy & Exclusion Verification Audit:")
for field in excluded_fields_check:
    is_present = field in df.columns
    print(f"Field '{field}': {'DANGER - PRESENT IN MATRIX' if is_present else 'SAFE - EXCLUDED'}")

print("\nFinal Clean Feature Vector Columns:", df.columns.tolist())

Privacy & Exclusion Verification Audit:
Field 'post_snapshot_impressions_90d': SAFE - EXCLUDED
Field 'client_tenant_id': SAFE - EXCLUDED
Field 'raw_target_url': SAFE - EXCLUDED
Field 'future_ga4_sessions': SAFE - EXCLUDED

Final Clean Feature Vector Columns: ['content_id', 'snapshot_date', 'is_available', 'impressions_30d', 'clicks_30d', 'ctr_30d', 'avg_position', 'ai_sessions_30d', 'ai_chatgpt_30d', 'ai_perplexity_30d', 'ai_gemini_30d', 'ai_copilot_30d', 'ai_claude_30d', 'word_count', 'content_age_days', 'is_declining']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.